In [1]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


Kaggle credentials set.
Kaggle credentials successfully validated.


In [2]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

hackeando_el_algoritmo_de_tik_tok_cetys_2026_2_path = kagglehub.competition_download('hackeando-el-algoritmo-de-tik-tok-cetys-2026-2')

print('Data source import complete.')


100%|██████████| 267k/267k [00:00<00:00, 81.2MB/s]

Extracting files...
Data source import complete.


In [3]:
# ==============================================================================
# 🚫 CELDA BLOQUEADA: EVALUADOR "CAJA NEGRA" (CHECKPOINTS LOCALES)
# Ejecuta esta celda al iniciar y no modifiques su contenido.
# ==============================================================================
import pandas as pd
import numpy as np

class EvaluadorKaggleFYP:
    @staticmethod
    def auditar_estrella(df_fact, df_dim_creadores, df_dim_audios):
        print("🔍 Evaluando Checkpoint 1: Esquema Estrella...")
        if not df_dim_creadores['usr_key'].is_unique: return "❌ ERROR: 'usr_key' tiene duplicados."
        if not df_dim_audios['snd_key'].is_unique: return "❌ ERROR: 'snd_key' tiene duplicados."

        columnas_prohibidas = ['attr_tier', 'attr_tenure_m', 'attr_bpm']
        for col in columnas_prohibidas:
            if col in df_fact.columns: return f"❌ ERROR: La columna descriptiva '{col}' sigue en fact_telemetria."

        print("✅ CHECKPOINT 1 SUPERADO: Normalización exitosa.\n")
        return True

    @staticmethod
    def auditar_cubo_olap(df_cubo):
        print("🔍 Evaluando Checkpoint 2: Cubo OLAP...")
        # En el set de entrenamiento hay ~700 videos orgánicos, en el de test hay ~300.
        # Si la llave es única, el roll-up se hizo bien.
        if 'vid_key' not in df_cubo.columns or not df_cubo['vid_key'].is_unique:
            return "❌ ERROR: 'vid_key' no es único. No agrupaste a nivel de video."

        print("✅ CHECKPOINT 2 SUPERADO: Granularidad OLAP correcta.\n")
        return True

evaluador = EvaluadorKaggleFYP()

# 📱 Hackeando el Algoritmo de TikTok (FYP)
Sigue estrictamente las reglas del **Diccionario de Telemetría** impreso. Recuerda que cualquier transformación que le apliques al set de Entrenamiento, deberás replicarla exactamente igual en el set de Prueba.

---

## Fase 1: Data Warehouse (Esquema Estrella)
**Objetivo:** Eliminar la redundancia.
Separa el archivo crudo (`train.csv`) en tres tablas (`df_dim_creadores`, `df_dim_audios`, y `df_fact_telemetria`) garantizando integridad referencial. Asegúrate de eliminar las columnas descriptivas de la tabla de hechos.

In [6]:
# NOTA: Si usas Kaggle directamente, la ruta de los archivos suele estar en '/kaggle/input/tu-competencia/'
df_train_raw = pd.read_csv('train.csv')
df_test_raw = pd.read_csv('test.csv')

print("TRAIN DATASET HEAD")
display(df_train_raw.head())

# --- TU CÓDIGO AQUÍ: Construye el Esquema Estrella (usa df_train_raw para la auditoría local) ---
df_dim_creadores = df_train_raw[['usr_key','attr_tier', 'attr_tenure_m']].drop_duplicates().reset_index(drop=True)
# df_dim_creadores['id_creador'] = df_dim_creadores.index + 1
df_dim_audios = df_train_raw[['snd_key','attr_bpm']].drop_duplicates().reset_index(drop=True)
# df_dim_audios['id_audio'] = df_dim_audios.index + 1
df_fact_telemetria = df_train_raw[['vid_key', 'usr_key', 'snd_key', 'sys_traffic_src', 'cat_code', 'pub_hour_utc', 'vid_len_sec', 'val_watch_sec', 'val_shares_cnt', 'val_swipe_pct']].drop_duplicates().reset_index(drop=True)
# df_fact_telemetria['id_telemetria'] = df_fact_telemetria.index + 1
# Validación del Checkpoint 1
evaluador.auditar_estrella(df_fact_telemetria, df_dim_creadores, df_dim_audios)

print("DIMENSION CREADORES")
display(df_dim_creadores.head())

print("DIMENSION AUDIOS")
display(df_dim_audios.head())

print("TABLA DE HECHOS")
display(df_fact_telemetria.head())

TRAIN DATASET HEAD


,log_event_id,vid_key,batch_hour_seq,sys_traffic_src,usr_key,attr_tier,attr_tenure_m,snd_key,attr_bpm,cat_code,pub_hour_utc,vid_len_sec,val_watch_sec,val_shares_cnt,val_swipe_pct,post_payout_usd,target_fyp_viral
0,EVT_101935,VID_00194,6,ORGANIC,USR_0141,Top_Partner,51,SND_0041,102,Storytime,11,15,5.30,20,0.635,0.00,0
1,EVT_106494,VID_00650,5,ORGANIC,USR_0216,Cuenta_Nueva,46,SND_0100,137,Storytime,5,15,-999.00,21,0.463,0.00,0
2,EVT_100360,VID_00037,1,ORGANIC,USR_0058,Cuenta_Nueva,28,SND_0127,154,Gaming,23,45,23.50,21,0.450,0.00,0
3,EVT_109663,VID_00967,4,ORGANIC,USR_0051,Top_Partner,46,SND_0041,102,Tech_Coding,14,30,11.37,14,0.597,0.00,0
4,EVT_108546,VID_00855,7,ORGANIC,USR_0140,Top_Partner,16,SND_0107,82,Storytime,16,45,29.08,16,0.374,166.73,1


🔍 Evaluando Checkpoint 1: Esquema Estrella...
✅ CHECKPOINT 1 SUPERADO: Normalización exitosa.

DIMENSION CREADORES


,usr_key,attr_tier,attr_tenure_m
0,USR_0141,Top_Partner,51
1,USR_0216,Cuenta_Nueva,46
2,USR_0058,Cuenta_Nueva,28
3,USR_0051,Top_Partner,46
4,USR_0140,Top_Partner,16


DIMENSION AUDIOS


,snd_key,attr_bpm
0,SND_0041,102
1,SND_0100,137
2,SND_0127,154
3,SND_0107,82
4,SND_0163,143


TABLA DE HECHOS


,vid_key,usr_key,snd_key,sys_traffic_src,cat_code,pub_hour_utc,vid_len_sec,val_watch_sec,val_shares_cnt,val_swipe_pct
0,VID_00194,USR_0141,SND_0041,ORGANIC,Storytime,11,15,5.30,20,0.635
1,VID_00650,USR_0216,SND_0100,ORGANIC,Storytime,5,15,-999.00,21,0.463
2,VID_00037,USR_0058,SND_0127,ORGANIC,Gaming,23,45,23.50,21,0.450
3,VID_00967,USR_0051,SND_0041,ORGANIC,Tech_Coding,14,30,11.37,14,0.597
4,VID_00855,USR_0140,SND_0107,ORGANIC,Storytime,16,45,29.08,16,0.374


## Fase 2: Operaciones OLAP (Cubo de Datos)
**Objetivo:** Limpieza y Granularidad.
1. Encuentra y repara los errores de los sensores de telemetría.
2. Filtra el tráfico ilegítimo (Bots).
3. Agrupa la telemetría transaccional para tener **1 sola fila por video**. Decide con inteligencia qué métricas se suman y cuáles se promedian.

In [7]:
  # --- TU CÓDIGO AQUÍ:
df_cubo_train = df_fact_telemetria[df_fact_telemetria['val_watch_sec'] > 0]
df_cubo_train = df_cubo_train[df_cubo_train['sys_traffic_src'] != 'BOT_FARM']

# print(len(df_cubo_train['vid_key']))
# print(len(df_cubo_train['vid_key'].unique()))

df_cubo_train = df_cubo_train.groupby('vid_key').agg({
    'val_watch_sec':'mean',
    'val_shares_cnt':'mean',
    'val_swipe_pct':'mean',
    'cat_code':'nunique',
    'usr_key':'nunique',
    'snd_key':'nunique',
}).sort_values('vid_key', ascending=False).reset_index()


display(df_cubo_train)

# Validación del Checkpoint 2
evaluador.auditar_cubo_olap(df_cubo_train)

,vid_key,val_watch_sec,val_shares_cnt,val_swipe_pct,cat_code,usr_key,snd_key
0,VID_01197,17.136250,19.250000,0.442000,1,1,1
1,VID_01196,21.761000,17.600000,0.258000,1,1,1
2,VID_01195,4.122500,19.250000,0.873375,1,1,1
3,VID_01194,16.850000,19.200000,0.468500,1,1,1
4,VID_01192,30.195556,14.000000,0.310000,1,1,1
...,...,...,...,...,...,...,...
691,VID_00006,13.345556,19.111111,0.700778,1,1,1
692,VID_00005,34.923000,15.600000,0.224300,1,1,1
693,VID_00004,7.948000,8.200000,0.490600,1,1,1
694,VID_00003,4.089000,12.200000,0.739300,1,1,1


🔍 Evaluando Checkpoint 2: Cubo OLAP...
✅ CHECKPOINT 2 SUPERADO: Granularidad OLAP correcta.



True

## Fase 3: Feature Engineering y Modelado
**Objetivo:** Traducir el negocio a matemáticas.
1. Une tus dimensiones de vuelta al cubo limpio.
2. Aplica codificación a las variables categóricas (cuidado con la memoria RAM, no uses One-Hot en columnas masivas).
3. Crea la métrica de negocio: **Hook Rate**.
4. Elimina IDs, variables originales transformadas y la trampa de *Data Leakage* antes de entrenar tu modelo.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# --- TU CÓDIGO AQUÍ: Feature Engineering  ---
X_train = pd.merge(df_cubo_train, df_dim_creadores, on='usr_key', how='left')
X_train = pd.merge(X_train, df_dim_audios, on='snd_key', how='left')
X_train = X_train.drop(columns=['vid_key'])
y_train = df_cubo_train['target_fyp_viral']
X_test_limpio = df_test_raw.drop(columns=['vid_key'])

# 1. Entrenar el Modelo
modelo = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42))
modelo.fit(X_train, y_train)

# 2. Predecir sobre el set de prueba ciego
predicciones = modelo.predict(X_test_limpio)

# 3. Exportar a Kaggle
submission = pd.DataFrame({
    'vid_key': X_test_limpio['vid_key'], # Asegúrate de que tu matriz final aún tenga el ID o guárdalo antes
    'target_fyp_viral': predicciones
})
submission.to_csv('submission.csv', index=False)
print("¡Archivo listo! Súbelo al Leaderboard.")

ValueError: You are trying to merge on int64 and object columns for key 'usr_key'. If you wish to proceed you should use pd.concat